# 03 – Explore SCB municipality statistics (PxWebApi v2)

**Goal:** find the right SCB tables for population and land area per municipality (*kommun*), load them, and check that all 290 municipalities are there.

**Part A** finds the tables – run it first.
**Part B** checks the loaded data – run it after filling in `TABLES` in `scb_pxweb.py` and running `uv run python -m hamburgarkartan.scb_pxweb`.

---
# Part A – Find the tables
SCB has thousands of tables. We search by words, then look inside the best match.

In [1]:
from hamburgarkartan.scb_pxweb import search_tables, show_metadata

## A1. Population per municipality

In [2]:
# id, first–last period, title
search_tables("folkmängd kommun")

TAB4951   2015–2020  Förvärvsarbetande och folkmängd i och utanför tätorter, efter region. År 2015-2020
TAB4704   2000–2020  Folkmängd, landareal och andel fritidshusområden efter region och typ av område. Vart femte år 2000-2020
TAB4422   2013–2025  Folkmängd inom zon runt skyddad natur och medelavstånd till skyddad natur, efter region. År 2013-2025
TAB631    2005–2023  Folkmängd och landareal i och utanför tätorter, efter region. Vart femte år 2005-2023
TAB6646   2025–2025  Folkmängden efter region, födelseland och kön.  År 2025
TAB6030   2000–2024  Folkmängden efter region, födelseland och kön. År 2000-2024
TAB6063   2015–2023  Folkmängd i tätorter efter kön, ålder och region. Vart femte år 2015-2023
TAB629    2015–2023  Folkmängd i småorter efter kön, ålder och region. Vart femte år 2015-2023
TAB6653   2025–2025  Folkmängden per distrikt, landskap, landsdel eller riket efter kön.  År 2025
TAB4537   2015–2024  Folkmängden per distrikt, landskap, landsdel eller riket efter kön. År 20

**Finding:** fill in – which table has population per municipality **for 2025**? (Candidate: `TAB5557`)

## A2. Land area per municipality

In [3]:
search_tables("landareal")

TAB2151   1992–1999  Landareal per den 1 januari efter region (uppdateras ej). År 1992-1999
TAB5396   2015–2025  Markägande i naturreservat, efter typ av markägare och storleksklass landareal . År 2015-2025
TAB4198   2015–2020  Byggnader, landareal och andel tätort i fritidshusområden efter region. Vart femte år 2015-2020
TAB3818   2015–2020  Byggnader, landareal och andel tätort i fritidshusområden efter fritidshusområde. Vart femte år 2015-2020
TAB2152   1990–2023  Folkmängd och landareal i småorter, per småort. Vart femte år 1990-2023
TAB4704   2000–2020  Folkmängd, landareal och andel fritidshusområden efter region och typ av område. Vart femte år 2000-2020
TAB631    2005–2023  Folkmängd och landareal i och utanför tätorter, efter region. Vart femte år 2005-2023
TAB5863   2015–2020  Antal verksamhetsområden och landareal, efter region. Vart femte år 2015-2020
TAB5357   1960–2023  Folkmängd och landareal i tätorter, per tätort. Vart femte år 1960-2023
TAB628    1991–2025  Befolkning

**Finding:** fill in – which table has land area per municipality, and for which years?

## A3. Look inside the chosen tables
Each table has *dimensions* (e.g. `Region`, `ContentsCode`, `Tid`). We need their codes to build the request.

In [4]:
# change the id to the population table you chose in A1
show_metadata("TAB5557")

Folkmängden efter region, civilstånd, ålder och kön. År 2025

Region  (312 values)
   00           Riket
   01           Stockholms län
   0114         Upplands Väsby
   0115         Vallentuna
   0117         Österåker
   0120         Värmdö
   0123         Järfälla
   0125         Ekerö
   ...

Civilstand  (5 values)
   SC           totalt, samtliga civilstånd
   OG           ogifta
   G            gifta
   SK           skilda
   ÄNKL         änkor/änklingar

Alder  (137 values)
   TotSA        totalt, samtliga åldrar
   0            0 år
   -4           0–4 år
   -9           0–9 år
   1            1 år
   2            2 år
   3            3 år
   4            4 år
   ...

Kon  (3 values)
   TotSa        totalt, samtliga män och kvinnor
   1            män
   2            kvinnor

ContentsCode  (2 values)
   000007ME     Folkmängd
   000007MG     Folkökning

Tid  (1 values)
   2025         2025


In [5]:
show_metadata("TAB628")

Befolkningstäthet (invånare per kvadratkilometer), folkmängd och landareal efter region och kön. År 1991-2025

Region  (312 values)
   00           Riket
   01           Stockholms län
   0114         Upplands Väsby
   0115         Vallentuna
   0117         Österåker
   0120         Värmdö
   0123         Järfälla
   0125         Ekerö
   ...

Kon  (3 values)
   1            män
   2            kvinnor
   1+2          totalt

ContentsCode  (3 values)
   BE0101U1     Invånare per kvadratkilometer
   BE0101U2     Folkmängd
   BE0101U3     Landareal i kvadratkilometer

Tid  (35 values)
   1991         1991
   1992         1992
   1993         1993
   1994         1994
   1995         1995
   1996         1996
   1997         1997
   1998         1998
   ...


**Finding:** fill in for each table – the dimension names, the `ContentsCode` we want, and whether 2025 exists.

Then write them into `TABLES` in `scb_pxweb.py`, e.g.:
```python
TABLES = {
    "population": ("TAB5557", {"Region": ["*"], "ContentsCode": ["<code>"], "Tid": ["2025"]}),
    "land_area":  ("TAB....", {"Region": ["*"], "ContentsCode": ["<code>"], "Tid": ["2025"]}),
}
```
and run `uv run python -m hamburgarkartan.scb_pxweb`.

---
# Part B – Check the loaded data

In [6]:
import duckdb
from hamburgarkartan.scb_pxweb import DB_PATH

# read_only = we only look, and other programs can still use the database
con = duckdb.connect(str(DB_PATH), read_only=True)

# which SCB tables were loaded?
con.sql("""
    SELECT table_name, estimated_size AS rows
    FROM duckdb_tables()
    WHERE schema_name = 'raw' AND table_name LIKE 'scb_%'
""")

┌──────────────┬────────┐
│  table_name  │  rows  │
│   varchar    │ int64  │
├──────────────┼────────┤
│ scb_grid_1km │ 115118 │
└──────────────┴────────┘

## B1. What do the rows look like?
Long format: one row per region × measure × year.

In [7]:
con.sql("SELECT * EXCLUDE (_loaded_at) FROM raw.scb_population LIMIT 5")

CatalogException: Catalog Error: Table with name scb_population does not exist!
Did you mean "pg_catalog.pg_collation"?

## B2. Which kinds of regions did we get?
Region codes tell the level: `00` = Sweden, 2 digits = county (*län*), 4 digits = municipality. We asked for all (`*`) and keep only the 4-digit ones later in dbt.
Codes must stay **text** – otherwise `0180` (Stockholm) becomes `180`.

In [ ]:
con.sql("""
    SELECT
        CASE length(Region)
            WHEN 2 THEN CASE WHEN Region = '00' THEN 'Sweden' ELSE 'county' END
            WHEN 4 THEN 'municipality'
            ELSE 'other'
        END AS level,
        count(*) AS rows,
        count(*) FILTER (WHERE value IS NULL) AS missing_values,
        typeof(any_value(Region)) AS code_type   -- should be VARCHAR
    FROM raw.scb_population
    GROUP BY 1
    ORDER BY 1
""")

**Finding:** fill in – exactly **290** municipalities, no missing values, codes stored as text?

## B3. Does the population add up?
The sum of the municipalities should equal the value for Sweden (`00`).

In [ ]:
con.sql("""
    SELECT
        sum(value) FILTER (WHERE length(Region) = 4) AS sum_municipalities,
        max(value) FILTER (WHERE Region = '00')       AS sweden,
        sum_municipalities - sweden                   AS difference   -- should be 0
    FROM raw.scb_population
""")

**Finding:** fill in – Sweden's population 2025 and the difference.

## B4. Compare with the 1 km grid
Both come from SCB, but the grid is adjusted for privacy – a small difference is expected.

In [ ]:
grid_cols = [c for c, t, *_ in con.sql("DESCRIBE raw.scb_grid_1km").fetchall()
             if t in ("BIGINT", "INTEGER", "DOUBLE", "HUGEINT") or t.startswith("DECIMAL")]
POP = next(c for c in grid_cols if any(k in c.lower() for k in ("beftot", "tot", "bef", "pop")))
print("grid population column:", POP)   # change by hand if wrong

con.sql(f"""
    WITH totals AS (
        SELECT
            (SELECT sum({POP}) FROM raw.scb_grid_1km)                       AS grid_total,
            (SELECT max(value) FROM raw.scb_population WHERE Region = '00') AS official_total
    )
    SELECT *, round(100 * (grid_total - official_total) / official_total, 2) AS diff_pct
    FROM totals
""")

**Finding:** fill in – how big is the difference in %? Under ~1 % is fine.

## B5. Largest and smallest municipalities

In [ ]:
con.sql("""
    (SELECT 'largest' AS kind, Region_label AS municipality, value AS population
     FROM raw.scb_population WHERE length(Region) = 4 ORDER BY value DESC LIMIT 5)
    UNION ALL
    (SELECT 'smallest', Region_label, value
     FROM raw.scb_population WHERE length(Region) = 4 ORDER BY value ASC LIMIT 5)
""")

## B6. Land area
Check the **unit** in the measure label (km² or hectares) – it matters for density.

In [ ]:
con.sql("""
    SELECT
        any_value(ContentsCode_label)               AS measure,   -- read the unit here
        count(*) FILTER (WHERE length(Region) = 4)  AS municipalities,
        round(sum(value) FILTER (WHERE length(Region) = 4)) AS total_land_area,
        min(value) FILTER (WHERE length(Region) = 4) AS smallest,
        max(value) FILTER (WHERE length(Region) = 4) AS largest
    FROM raw.scb_land_area
""")

**Finding:** fill in – unit, and total land area (Sweden has roughly 407 000 km² of land).

## B7. Population density – a first look
Joins the two tables on the municipality code. This is the kind of join dbt will do later.

In [ ]:
con.sql("""
    SELECT
        p.Region        AS code,
        p.Region_label  AS municipality,
        p.value         AS population,
        a.value         AS land_area,
        round(p.value / a.value, 1) AS people_per_area_unit
    FROM raw.scb_population p
    JOIN raw.scb_land_area a USING (Region)
    WHERE length(p.Region) = 4
    ORDER BY people_per_area_unit DESC
    LIMIT 10
""")

**Finding:** fill in – do all 290 municipalities join? The densest should be around Stockholm.

## Summary
- Tables used: population `TAB____`, land area `TAB____` (year 2025)
- Municipalities: _fill in_ (should be 290)
- Population 2025: _fill in_; grid vs official difference: _fill in_ %
- Land area unit: _fill in_

**Next:** 4d – municipality boundaries (polygons), so every grid cell and restaurant can be placed in a municipality.

In [ ]:
con.close()